# 02 – Indeksi dhe chunk-et (Detyra B)
Projekti 06 • Java 2 • Roli B (Klara Belulaj), zëvendësues Gerald Paja

**Qëllimi:** abstraktet ndahen në fragmente (chunks) me ID të gjurmueshme, krijohen embeddings me model të
gatshëm dhe ndërtohet një indeks vektorial (FAISS). Testohet që një fragment i gjetur hapet si tekst dhe i
përket `source_id` të regjistruar.

> Kufizim: provë teknike mësimore. Pa të dhëna pacientësh, pa diagnozë ose këshillë mjekësore.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
sys.path.insert(0, str(Path("..").resolve() / "src"))
import minirag as mr
pd.set_option("display.max_colwidth", 90); pd.set_option("display.width", 200)

# Korpusi: REAL06 nëse ekzistojnë abstraktet reale (pas `python src/merr_abstraktet.py`), përndryshe SIM06.
REAL_PATH = mr.ROOT / "data/private/02_Abstraktet_Reale.csv"
CORPUS = "REAL06" if REAL_PATH.exists() else "SIM06"
print("Korpusi aktiv:", CORPUS)

Korpusi aktiv: REAL06


## 1. Ngarkimi i korpusit dhe lidhja me manifestin

In [2]:
docs = mr.load_real_corpus() if CORPUS == "REAL06" else mr.load_sim_corpus()
print(f"{len(docs)} dokumente")
print(docs[["source_id", "title", "license_status", "shareable"]].to_string(index=False))

manifest = pd.read_csv(mr.ROOT / "data/manifests/02_Korpusi_Pilot_Manifest.csv")
print(f"\nManifesti real ka {len(manifest)} burime; versioni: {manifest.corpus_version.iloc[0]}")

Traceback (most recent call last):
  File "/home/claude/projekti06/tools/build_notebooks.py", line 43, in execute
    exec(compile(c["source"], f"{name}[{n}]", "exec"), ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "02_Indeksi_Dhe_Chunket.ipynb[2]", line 1, in <module>
  File "/home/claude/projekti06/src/minirag.py", line 64, in load_real_corpus
    df = pd.read_csv(path)
  File "/usr/local/lib/python3.13/site-packages/pandas/io/parsers/readers.py", line 873, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "/usr/local/lib/python3.13/site-packages/pandas/io/parsers/readers.py", line 300, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "/usr/local/lib/python3.13/site-packages/pandas/io/parsers/readers.py", line 1645, in __init__
    self._engine = self._make_engine(f, self.engine)
                   ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/site-packages/pandas/io/parsers/readers.py", line 1922, in _m

## 2. Chunking
- **SIM06:** çdo dokument është tashmë një fragment i shkurtër → 1 chunk/dokument.
- **REAL06:** abstrakti ndahet në grupe me 1–2 fjali (maks. ~600 karaktere), pa mbivendosje.

Çdo chunk ruan `chunk_id = <source_id>-Cnn`, `start_offset`/`end_offset` te teksti origjinal dhe versionet e
pastrimit/segmentimit.

In [3]:
chunks = mr.build_chunks(docs, single_chunk=(CORPUS == "SIM06"))
print(f"{len(chunks)} chunk-e  |  versioni: {chunks.chunking_version.iloc[0]}")
print(chunks[["chunk_id", "source_id", "start_offset", "end_offset", "n_chars"]].head(12).to_string(index=False))

Traceback (most recent call last):
  File "/home/claude/projekti06/tools/build_notebooks.py", line 43, in execute
    exec(compile(c["source"], f"{name}[{n}]", "exec"), ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "02_Indeksi_Dhe_Chunket.ipynb[3]", line 1, in <module>
NameError: name 'docs' is not defined


### Kontrolli i gjurmueshmërisë: `text[start:end]` i burimit duhet të jetë identik me chunk-un

In [4]:
errors = mr.verify_chunk_provenance(chunks, docs)
print("Gabime gjurmueshmërie:", errors if errors else "asnjë ✔")

Traceback (most recent call last):
  File "/home/claude/projekti06/tools/build_notebooks.py", line 43, in execute
    exec(compile(c["source"], f"{name}[{n}]", "exec"), ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "02_Indeksi_Dhe_Chunket.ipynb[4]", line 1, in <module>
NameError: name 'chunks' is not defined


## 3. Embeddings dhe indeksi vektorial

In [5]:
rag = mr.MiniRAG.build(chunks)
print("Embedder:", rag.embedder.backend)
print("Indeksi: ", rag.index.backend, "| vektorë:", len(rag.index), "| dimensioni:", rag.index.dim)
if not rag.embedder.is_semantic:
    print("\nKUJDES: sentence-transformers nuk është instaluar në këtë mjedis -> u përdor TF-IDF (fallback).")
    print("Në mjedisin e ekipit: pip install -r requirements.txt dhe riekzekutoni notebook-un.")

Traceback (most recent call last):
  File "/home/claude/projekti06/tools/build_notebooks.py", line 43, in execute
    exec(compile(c["source"], f"{name}[{n}]", "exec"), ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "02_Indeksi_Dhe_Chunket.ipynb[5]", line 1, in <module>
NameError: name 'chunks' is not defined


## 4. Shembull përputhjeje: një pyetje → fragmenti më i afërt hapet si tekst

In [6]:
q = ("Cili rezultat është regjistruar në momentin e daljes nga spitali?" if CORPUS == "SIM06"
     else "What was the crude annual incidence of aneurysmal subarachnoid hemorrhage in Norway between 2008 and 2014?")
top = rag.retrieve(q, k=3)
print("Pyetja:", q, "\n")
for r in top.itertuples():
    print(f"#{r.rank}  {r.chunk_id}  (score {r.retrieval_score})  burimi={r.source_id}")
    print("    ", r.chunk_text[:220] + ("…" if len(r.chunk_text) > 220 else ""))

best = top.iloc[0]
src_text = docs.set_index("source_id").loc[best.source_id, "text"]
row = chunks.set_index("chunk_id").loc[best.chunk_id]
assert src_text[row.start_offset:row.end_offset] == best.chunk_text
print(f"\n✔ {best.chunk_id} i përket {best.source_id} dhe gjendet te pozicioni {row.start_offset}–{row.end_offset}.")

Traceback (most recent call last):
  File "/home/claude/projekti06/tools/build_notebooks.py", line 43, in execute
    exec(compile(c["source"], f"{name}[{n}]", "exec"), ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "02_Indeksi_Dhe_Chunket.ipynb[6]", line 3, in <module>
NameError: name 'rag' is not defined


## 5. Ruajtja e `02_Chunket_Pilot.csv`
Teksti i burimeve pa licencë të hapur nuk shkruhet në repository.

In [7]:
mr.upsert_csv(mr.export_chunks(chunks), mr.ROOT / "results/02_Chunket_Pilot.csv", CORPUS)
saved = pd.read_csv(mr.ROOT / "results/02_Chunket_Pilot.csv")
print(saved.groupby("corpus").size().to_string())

Traceback (most recent call last):
  File "/home/claude/projekti06/tools/build_notebooks.py", line 43, in execute
    exec(compile(c["source"], f"{name}[{n}]", "exec"), ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "02_Indeksi_Dhe_Chunket.ipynb[7]", line 1, in <module>
NameError: name 'chunks' is not defined


## 6. Zgjedhjet teknike dhe kufizimet
| Zgjedhja | Arsyeja | Kufizimi |
|---|---|---|
| `paraphrase-multilingual-MiniLM-L12-v2` | model i gatshëm, i vogël, shumëgjuhësh (pyetje shqip ↔ abstrakte anglisht) | jo i specializuar për mjekësi |
| FAISS `IndexFlatIP` | kërkim i saktë (jo i përafërt) për pak vektorë | për korpus të madh duhet indeks i përafërt |
| 1–2 fjali/chunk | fragmente të shkurtra → citime të sakta | konteksti mund të ndahet mes dy chunk-eve |
| TF-IDF (fallback) | punon pa internet | leksikor: nuk kap sinonime dhe nuk kalon gjuhë |